In [24]:
from datetime import datetime, timedelta, timezone
import github3
import os
from pathlib import Path
import sys
import time

In [17]:
REPO = 'bytecodealliance/wasm-micro-runtime'

*Please follow https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens to get a proper token.*

In [ ]:
TOKEN_PATH = Path('./token')


In [35]:
PROXIES = {
    "http": "http://childprc-intel.com:913",
    "https": "http://childprc-intel.com:913"
}

In [ ]:

def get_token(token_path):
    """
    Retrieve the GitHub token from a file or prompt the user to create it.
    """
    assert token_path.exists(), f"Token file not found. Please acquire a token, and store it at {token_path}"

    # if token is empty
    with open(token_path, 'r') as f:
        token = f.read().strip()
        assert token, f"Token file is empty. Please acquire a token, and store it at {token_path}"

        return token

In [29]:
def print_error_messages(error: github3.exceptions):
    """Prints the error messages from the GitHub API response.

    Args:
        Error (github3.exceptions): The error object from the GitHub API response.

    """
    if hasattr(error, "errors"):
        for e in error.errors:
            print(f"Error: {e.get('message')}")

In [53]:
def setup_connection(repo_full_name, token_path):
    """
    Make a connection to the specified GitHub repository.
    """
    token = get_token(token_path)
    gh = github3.GitHub(token=token)
    gh.session.proxies['http://'] = 'http://child-prc.intel.com:913'
    gh.session.proxies['https://'] = 'http://child-prc.intel.com:913'

    # a smoke test on connection
    current_user = gh.me()
    assert current_user.name == "liang.he", f"Expected user 'liang.he', but got '{current_user.name}'. Might want to change the token"

    return gh


In [54]:
def search_issues(github_connection, repos_and_owners_string, search_query, rate_limit_bypass = False):
    # Rate Limit Handling: API only allows 30 requests per minute
    def wait_for_api_refresh(
        iterator: github3.structs.SearchIterator, rate_limit_bypass: bool = False
    ):
        # If the rate limit bypass is enabled, don't wait for the API to refresh
        if rate_limit_bypass:
            return

        max_retries = 5
        retry_count = 0
        sleep_time = 70

        while iterator.ratelimit_remaining < 5:
            if retry_count >= max_retries:
                raise RuntimeError("Exceeded maximum retries for API rate limit")

            print(
                f"GitHub API Rate Limit Low, waiting {sleep_time} seconds to refresh."
            )
            time.sleep(sleep_time)

            # Exponentially increase the sleep time for the next retry
            sleep_time *= 2
            retry_count += 1

    print(f"Searching for issues... with {search_query} in {repos_and_owners_string}")
    issues_per_page = 100
    issues_iterator = github_connection.search_issues(
        search_query, per_page=issues_per_page
    )
    wait_for_api_refresh(issues_iterator, rate_limit_bypass)

    issues = []
    # Print the issue titles and add them to the list of issues
    try:
        for idx, issue in enumerate(issues_iterator, 1):
            print(issue.title)  # type: ignore
            issues.append(issue)

            # requests are sent once per page of issues
            if idx % issues_per_page == 0:
                wait_for_api_refresh(issues_iterator, rate_limit_bypass)

    except github3.exceptions.ForbiddenError as e:
        print(
            f"You do not have permission to view a repository \
from: '{repos_and_owners_string}'; Check your API Token."
        )
        print_error_messages(e)
        sys.exit(1)
    except github3.exceptions.NotFoundError as e:
        print(
            f"The repository could not be found; \
Check the repository owner and names: '{repos_and_owners_string}"
        )
        print_error_messages(e)
        sys.exit(1)
    except github3.exceptions.ConnectionError as e:
        print(
            "There was a connection error; Check your internet connection or API Token."
        )
        print_error_messages(e)
        sys.exit(1)
    except github3.exceptions.AuthenticationFailed as e:
        print("Authentication failed; Check your API Token.")
        print_error_messages(e)
        sys.exit(1)
    except github3.exceptions.UnprocessableEntity as e:
        print("The search query is invalid; Check the search query.")
        print_error_messages(e)
        sys.exit(1)

    return issues

def search_issues_created_in_last_90days(github_connection, repos_and_owners_string, search_query, rate_limit_bypass = False):
    """
    Search for issues in the specified GitHub repository within the last 90 days.

    Args:
        github_connection: The authenticated GitHub connection.

    Returns:
        list: A list of issues matching the search query.
    """
    now = datetime.now(timezone.utc)
    since = now - timedelta(days=90)
    search_query = f"{search_query} created:>={since.isoformat()}"

    return search_issues(github_connection, repos_and_owners_string, search_query, rate_limit_bypass)

In [57]:
gh = setup_connection(REPO, TOKEN_PATH)
issues = search_issues_created_in_last_90days(gh, REPO, f"repo:{REPO} is:issue", False)

Searching for issues... with repo:bytecodealliance/wasm-micro-runtime is:issue created:>=2025-05-20T12:13:44.378136+00:00 in bytecodealliance/wasm-micro-runtime
Should we add a watchdog to prevent the similar crash in AOT mode?
Proposal: Add Github Actions workflow to auto-close stale issues
Should we have a standalone AOT validator?
lib-socket does not support getsockopt(SOL_SOCKET, SO_ERROR)
WAMR AOT and JIT modes hang indefinitely
How to handle `ref.null 0`?
WAMR AOT mode crashes
wasi-nn: missing address validation for get_output tensor buffer
WAMR outputs `LLVM ERROR`
WAMR AOT and JIT modes output `out of bounds` exception
WAMR AOT incorrectly outputs `out of bounds` exception
WAMR runtime incorrectly reports invalid
WAMR runtime behaves differently
WAMR_BUILD_WASI_NN_TFLITE doesn't seem compatible with cmake4
Incorrect start function signature validation
How to build WAMR for RP2350 (thumbv8m.main-none-eabihf) target
extended const should have bumped AOT_CURRENT_VERSION
Missing Su